# TP — Module 1 : Fondements Python pour l'Analyse Géospatiale
## CORRECTION COMPLÈTE

> **Cours** : ASML | **Institut** : 2iE | **Version** : Correction annotée

---
> Chaque bloc de correction inclut un commentaire `# 💡` expliquant le choix
> et le piège associé, en correspondance avec le CM.

## Environnement

> **Google Colab** : décommenter les lignes `!pip install` ci-dessous.

In [ ]:
# Décommenter si nécessaire (Google Colab)
# !pip install geopandas rasterio folium contextily matplotlib --quiet

### Imports

In [ ]:
import json, io, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import rasterio
from rasterio.transform import from_bounds
from rasterstats import zonal_stats
import folium
import contextily as ctx
import requests

print('Bibliothèques chargées. GeoPandas :', gpd.__version__)

# ═══════════════════════════════════════════════════════════════
# Données fallback — définies EN PREMIER (évite NameError)
# ═══════════════════════════════════════════════════════════════

GJ_REG_STR = '{"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"region": "Boucle du Mouhoun", "chef_lieu": "Dédougou", "population": 1427000, "superficie_ref": 34497}, "geometry": {"type": "Polygon", "coordinates": [[[-3.45, 12.15], [-2.3499999999999996, 12.15], [-2.3499999999999996, 13.049999999999999], [-3.45, 13.049999999999999], [-3.45, 12.15]]]}}, {"type": "Feature", "properties": {"region": "Cascades", "chef_lieu": "Banfora", "population": 730000, "superficie_ref": 18236}, "geometry": {"type": "Polygon", "coordinates": [[[-5.0, 10.05], [-3.9000000000000004, 10.05], [-3.9000000000000004, 10.95], [-5.0, 10.95], [-5.0, 10.05]]]}}, {"type": "Feature", "properties": {"region": "Centre", "chef_lieu": "Ouagadougou", "population": 2415000, "superficie_ref": 2585}, "geometry": {"type": "Polygon", "coordinates": [[[-2.0700000000000003, 11.91], [-0.97, 11.91], [-0.97, 12.809999999999999], [-2.0700000000000003, 12.809999999999999], [-2.0700000000000003, 11.91]]]}}, {"type": "Feature", "properties": {"region": "Centre-Est", "chef_lieu": "Tenkodogo", "population": 1430000, "superficie_ref": 14442}, "geometry": {"type": "Polygon", "coordinates": [[[-0.35000000000000003, 11.4], [0.75, 11.4], [0.75, 12.299999999999999], [-0.35000000000000003, 12.299999999999999], [-0.35000000000000003, 11.4]]]}}, {"type": "Feature", "properties": {"region": "Centre-Nord", "chef_lieu": "Kaya", "population": 1400000, "superficie_ref": 19889}, "geometry": {"type": "Polygon", "coordinates": [[[-1.6, 12.600000000000001], [-0.5, 12.600000000000001], [-0.5, 13.5], [-1.6, 13.5], [-1.6, 12.600000000000001]]]}}, {"type": "Feature", "properties": {"region": "Centre-Ouest", "chef_lieu": "Koudougou", "population": 1360000, "superficie_ref": 21543}, "geometry": {"type": "Polygon", "coordinates": [[[-3.0, 11.65], [-1.9000000000000001, 11.65], [-1.9000000000000001, 12.549999999999999], [-3.0, 12.549999999999999], [-3.0, 11.65]]]}}, {"type": "Feature", "properties": {"region": "Centre-Sud", "chef_lieu": "Manga", "population": 730000, "superficie_ref": 11731}, "geometry": {"type": "Polygon", "coordinates": [[[-1.6, 11.05], [-0.5, 11.05], [-0.5, 11.95], [-1.6, 11.95], [-1.6, 11.05]]]}}, {"type": "Feature", "properties": {"region": "Est", "chef_lieu": "Fada N\'Gourma", "population": 1580000, "superficie_ref": 47245}, "geometry": {"type": "Polygon", "coordinates": [[[0.8, 11.65], [1.9000000000000001, 11.65], [1.9000000000000001, 12.549999999999999], [0.8, 12.549999999999999], [0.8, 11.65]]]}}, {"type": "Feature", "properties": {"region": "Hauts-Bassins", "chef_lieu": "Bobo-Dioulasso", "population": 2040000, "superficie_ref": 25739}, "geometry": {"type": "Polygon", "coordinates": [[[-4.8, 10.700000000000001], [-3.7, 10.700000000000001], [-3.7, 11.6], [-4.8, 11.6], [-4.8, 10.700000000000001]]]}}, {"type": "Feature", "properties": {"region": "Nord", "chef_lieu": "Ouahigouya", "population": 1110000, "superficie_ref": 16855}, "geometry": {"type": "Polygon", "coordinates": [[[-2.9699999999999998, 13.13], [-1.8699999999999999, 13.13], [-1.8699999999999999, 14.03], [-2.9699999999999998, 14.03], [-2.9699999999999998, 13.13]]]}}, {"type": "Feature", "properties": {"region": "Plateau Central", "chef_lieu": "Ziniaré", "population": 790000, "superficie_ref": 8018}, "geometry": {"type": "Polygon", "coordinates": [[[-1.17, 12.100000000000001], [-0.06999999999999995, 12.100000000000001], [-0.06999999999999995, 13.0], [-1.17, 13.0], [-1.17, 12.100000000000001]]]}}, {"type": "Feature", "properties": {"region": "Sahel", "chef_lieu": "Dori", "population": 1230000, "superficie_ref": 44290}, "geometry": {"type": "Polygon", "coordinates": [[[-0.6000000000000001, 13.600000000000001], [0.5, 13.600000000000001], [0.5, 14.5], [-0.6000000000000001, 14.5], [-0.6000000000000001, 13.600000000000001]]]}}, {"type": "Feature", "properties": {"region": "Sud-Ouest", "chef_lieu": "Diébougou", "population": 730000, "superficie_ref": 16126}, "geometry": {"type": "Polygon", "coordinates": [[[-3.8, 10.15], [-2.7, 10.15], [-2.7, 11.049999999999999], [-3.8, 11.049999999999999], [-3.8, 10.15]]]}}]}'

fallback_gdf = gpd.GeoDataFrame.from_features(
    json.loads(GJ_REG_STR)['features'], crs='EPSG:4326'
)

GJ_EAU_STR = 'None'

puits = gpd.GeoDataFrame.from_features(
    json.loads(GJ_EAU_STR)['features'], crs='EPSG:4326'
)

print(f'Fallback : {len(fallback_gdf)} régions + {len(puits)} points d\'eau')

# ═══════════════════════════════════════════════════════════════
# Chargement GADM 4.1 — 13 vraies régions BF
# Source : https://gadm.org/download_country.html → GeoJSON level-1
# URL    : geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_BFA_1.json
# Fallback automatique si GADM inaccessible
# ═══════════════════════════════════════════════════════════════

URL_GADM_L1 = 'https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_BFA_1.json'

def load_gadm_regions(fallback):
    try:
        print('Chargement GADM level-1 (vraies géométries BF)...')
        r = requests.get(URL_GADM_L1, timeout=15)
        r.raise_for_status()
        gdf_raw = gpd.read_file(io.BytesIO(r.content))
        gdf_raw = gdf_raw.rename(columns={
            'NAME_0':'pays', 'NAME_1':'region', 'GID_1':'gid_region'
        })
        for col in ['chef_lieu','population','superficie_ref']:
            if col in fallback.columns and col not in gdf_raw.columns:
                mapping = fallback.set_index('region')[col].to_dict()
                gdf_raw[col] = gdf_raw['region'].map(mapping)
        print(f'[GADM ✅] {len(gdf_raw)} régions | CRS: {gdf_raw.crs}')
        return gdf_raw, 'GADM 4.1'
    except Exception as e:
        warnings.warn(f'GADM inaccessible ({e}) → fallback', stacklevel=2)
        print('[FALLBACK ⚠️] Rectangles embarqués utilisés')
        return fallback.copy(), 'fallback'

gdf, source_gdf = load_gadm_regions(fallback_gdf)

print(f'\nSource  : {source_gdf}')
print(f'Régions : {len(gdf)} | CRS : {gdf.crs}')


---
## Partie 1 — GeoDataFrame et Exploration des Données

> **Référence CM** : §3.1 Le GeoDataFrame, §3.2 Lecture des fichiers spatiaux

### 1.1 Chargement des données

Les deux jeux de données utilisés dans ce TP sont embarqués directement :
- `gdf` : 13 régions administratives du Burkina Faso (polygones)
- `puits` : 30 points d'eau — forages et puits AEPS/PEA (points)

### 1.2 Exercice 1 — Exploration

En suivant le pattern du CM (§3.1), répondez aux questions suivantes :

1. Affichez les **3 premières lignes** du GeoDataFrame `gdf`
2. Affichez les **types de colonnes** (`dtypes`)
3. Vérifiez que le type géométrique est bien **Polygon** pour toutes les régions
4. Quelle région a la **population la plus élevée** ? La plus faible ?
5. Combien y a-t-il de **forages** et de **puits** dans `puits` ?

In [ ]:
# ── Correction 1.1 : Aperçu ──
print('=== 3 premières lignes ===')
print(gdf[['region', 'chef_lieu', 'population', 'superficie_ref', 'geometry']].head(3))

# ── Correction 1.2 : Types ──
print('\n=== Types de colonnes ===')
print(gdf.dtypes)

# ── Correction 1.3 : Type géométrique ──
print('\n=== Types géométriques ===')
print(gdf.geometry.geom_type.value_counts())
# → Polygon 13  (toutes les régions sont des polygones)

# ── Correction 1.4 : Pop max / min ──
pop_max = gdf.loc[gdf['population'].idxmax(), ['region', 'population']]
pop_min = gdf.loc[gdf['population'].idxmin(), ['region', 'population']]
print('\nRégion la plus peuplée :', pop_max['region'], '—', f"{pop_max['population']:,} hab.")
print('Région la moins peuplée:', pop_min['region'], '—', f"{pop_min['population']:,} hab.")

# ── Correction 1.5 : Décompte types points ──
print('\n=== Types de points d\'eau ===')
print(puits['type'].value_counts())

# 💡 La règle d'or du CM (§3.1) : toujours commencer par
# shape, dtypes, crs et geom_type.value_counts().
# Ces 4 contrôles en 30 secondes évitent des heures de débogage.

---
## Partie 2 — Attributs Géométriques et Projections

> **Référence CM** : §3.3 Attributs géométriques, §5 Systèmes de référence

Le CM insiste sur une **règle absolue** :
> *Vérifiez toujours le CRS avant de calculer une superficie ou une distance.*
> *En EPSG:4326 (degrés), `.area` donne des degrés carrés — sans sens physique.*

### Exercice 2 — Métriques géométriques

1. Montrez que `gdf.geometry.area.mean()` en EPSG:4326 donne une valeur absurde
2. Reprojetez en **EPSG:32630** (UTM Zone 30N) et calculez :
   - `superficie_km2` (km²)
   - `perimetre_km` (km)
3. Calculez la **distance de chaque région au centroïde d'Ouagadougou** (en km)
   - Ouagadougou : lon = -1.5243, lat = 12.3647
4. Comparez `superficie_km2` calculée avec `superficie_ref` (données officielles INSD) :
   quel est l'écart moyen en % ? Expliquez pourquoi.

In [ ]:
# ── Correction 2.1 : Superficie en degrés² (absurde) ──
print('Superficie en degrés² (absurde) :', round(gdf.geometry.area.mean(), 4))
# → ~0.47  (degrés carrés — aucune signification physique)

# ── Correction 2.2 : Reprojection UTM 30N ──
gdf_utm = gdf.to_crs(epsg=32630)
gdf['superficie_km2'] = gdf_utm.geometry.area   / 1e6   # m² → km²
gdf['perimetre_km']   = gdf_utm.geometry.length / 1e3   # m  → km

print('\n=== Superficies et périmètres (UTM 30N) ===')
print(gdf[['region', 'superficie_km2', 'perimetre_km']]
        .sort_values('superficie_km2', ascending=False)
        .to_string(index=False))

# ── Correction 2.3 : Distance à Ouagadougou ──
# Créer un GeoSeries pour Ouaga, reprojeter en UTM 30N
ouaga_wgs = gpd.GeoSeries([Point(-1.5243, 12.3647)], crs=4326)
ouaga_utm = ouaga_wgs.to_crs(32630).iloc[0]

gdf['dist_ouaga_km'] = gdf_utm.geometry.centroid.distance(ouaga_utm) / 1e3

print('\n=== Distance des régions à Ouagadougou ===')
print(gdf[['region', 'dist_ouaga_km']].sort_values('dist_ouaga_km').to_string(index=False))

# ── Correction 2.4 : Écart superficie calculée vs officielle ──
gdf['ecart_pct'] = abs(gdf['superficie_km2'] - gdf['superficie_ref']) / gdf['superficie_ref'] * 100
print('\nÉcart moyen :', round(gdf['ecart_pct'].mean(), 1), '%')
print('(Normal : nos polygones sont des rectangles simplifiés,',
      'pas les vrais contours administratifs)')

# 💡 L'écart (20-60%) vient des géométries rectangulaires simplifiées.
# Avec les vrais shapefiles (Drive 1MHY8PprO8_...), l'écart < 1%.
# Ce test montre l'importance des données géométriques précises.

---
## Partie 3 — Jointures Spatiales

> **Référence CM** : §3.4 Jointures spatiales — `gpd.sjoin`

Le CM présente le cas : *quelle région contient chaque point d'eau ?*
C'est exactement ce que vous allez réaliser avec nos 30 points d'eau.

### Exercice 3 — Analyser la couverture en eau potable

1. Vérifiez que `gdf` et `puits` ont le **même CRS** avant la jointure
2. Réalisez une jointure spatiale `gpd.sjoin` pour savoir dans quelle région
   se trouve chaque point d'eau (`predicate='within'`)
3. Calculez le **nombre de points d'eau par région** et ajoutez la colonne
   `nb_points_eau` au GeoDataFrame `gdf`
4. Calculez le **ratio points d'eau / million d'habitants** (`ratio_peau_M`)
5. Identifiez la région la mieux et la moins bien équipée en eau potable

> ⚠️ Rappel CM : *les deux GeoDataFrames doivent être dans le même CRS*
> *avant une jointure spatiale — sinon les résultats sont silencieusement faux.*

In [ ]:
# ── Correction 3.1 : Vérification CRS ──
print('CRS régions :', gdf.crs)
print('CRS puits   :', puits.crs)
print('CRS identiques :', gdf.crs == puits.crs)

# Les deux sont en EPSG:4326 → on peut faire la jointure directement.
# Si différents : puits = puits.to_crs(gdf.crs) avant sjoin.

# ── Correction 3.2 : Jointure spatiale ──
# Pour chaque point d'eau, trouver la région qui le contient
joined = gpd.sjoin(puits, gdf, how='left', predicate='within')

print('\nAperçu jointure :')
print(joined[['nom_point', 'type', 'region']].head(8).to_string(index=False))
print('Points sans région (hors BF) :', joined['region'].isna().sum())

# ── Correction 3.3 : Comptage + merge ──
nb_peau = joined.groupby('region')['nom_point'].count().reset_index()
nb_peau.columns = ['region', 'nb_points_eau']

# Merge dans gdf (left join pour garder toutes les régions, même sans puits)
gdf = gdf.merge(nb_peau, on='region', how='left')
gdf['nb_points_eau'] = gdf['nb_points_eau'].fillna(0).astype(int)

print('\n=== Points d\'eau par région ===')
print(gdf[['region', 'nb_points_eau']].sort_values('nb_points_eau', ascending=False).to_string(index=False))

# ── Correction 3.4 : Ratio / million d'habitants ──
gdf['ratio_peau_M'] = gdf['nb_points_eau'] / (gdf['population'] / 1e6)

# ── Correction 3.5 : Meilleures / moins bonnes ──
meilleure = gdf.loc[gdf['ratio_peau_M'].idxmax(), ['region', 'nb_points_eau', 'ratio_peau_M']]
pire      = gdf.loc[gdf['ratio_peau_M'].idxmin(), ['region', 'nb_points_eau', 'ratio_peau_M']]
print('\nMieux équipée :', meilleure['region'],
      f"({meilleure['nb_points_eau']} pts, ratio {meilleure['ratio_peau_M']:.1f}/M hab.)")
print('Moins équipée :', pire['region'],
      f"({pire['nb_points_eau']} pts, ratio {pire['ratio_peau_M']:.1f}/M hab.)")

# 💡 Pourquoi left join et non inner join ?
# inner join élimine les régions sans aucun point d'eau → données manquantes cachées.
# left join conserve toutes les régions et attribue 0 aux non-couvertes.
# fillna(0) est essentiel après le merge pour éviter des NaN dans nb_points_eau.

---
## Partie 4 — Données Raster et Calcul du NDVI

> **Référence CM** : §4.1 Ouvrir et lire un raster, §4.2 Calcul du NDVI

Dans un projet réel, vous chargeriez des images Sentinel-2 depuis Copernicus
ou Google Earth Engine. Dans ce TP, nous générons un raster **synthétique**
qui reproduit le gradient bioclimatique Nord-Sud du Burkina Faso :
zones arides au Nord (Sahel, NDVI ~ 0.05), zones humides au Sud (NDVI ~ 0.60).

### 4.1 Génération et écriture du raster synthétique

> **Rappel CM §4.1** : Rasterio gère les rasters comme un livre —
> on l'ouvre avec `with rasterio.open()`, on lit les bandes (indices commençant à 1),
> puis on le ferme automatiquement à la sortie du bloc `with`.

**Exercice 4a** — Exécutez la cellule ci-dessous pour créer `bf_ndvi_synth.tif`.
Lisez le code et répondez : pourquoi utilise-t-on `.astype(float)` ?

In [ ]:
import rasterio
from rasterio.transform import from_bounds

# ── Paramètres du raster synthétique ──
# Bbox du Burkina Faso : lon -5.5 → 2.4 | lat 9.4 → 15.1
BBOX    = (-5.5, 9.4, 2.4, 15.1)   # (west, south, east, north)
ROWS, COLS = 100, 130               # Résolution pédagogique
NODATA  = -9999.0

# ── Simulation du gradient bioclimatique Nord-Sud ──
# lat augmente vers le Nord → gradient décroissant du Sud (NDVI élevé) vers le Nord
lats = np.linspace(BBOX[3], BBOX[1], ROWS)   # du Nord (row 0) vers le Sud (row -1)
lons = np.linspace(BBOX[0], BBOX[2], COLS)
LON, LAT = np.meshgrid(lons, lats)

# Gradient latitudinal : lat 15 → NDVI 0.05 | lat 9.4 → NDVI 0.62
ndvi_base = 0.62 - (LAT - 9.4) * (0.62 - 0.05) / (15.1 - 9.4)

# Bruit spatial réaliste
np.random.seed(42)
bruit = np.random.normal(0, 0.04, (ROWS, COLS))
ndvi  = np.clip(ndvi_base + bruit, -0.1, 0.9).astype(np.float32)

# ── Écriture GeoTIFF via Rasterio ──
transform = from_bounds(*BBOX, width=COLS, height=ROWS)

profile = {
    'driver': 'GTiff',
    'dtype': rasterio.float32,
    'width': COLS,
    'height': ROWS,
    'count': 1,          # 1 seule bande (le NDVI)
    'crs': 'EPSG:4326',
    'transform': transform,
    'nodata': NODATA
}

RASTER_PATH = 'bf_ndvi_synth.tif'
with rasterio.open(RASTER_PATH, 'w', **profile) as dst:
    dst.write(ndvi, 1)   # bande 1 (indices commencent à 1, pas 0 !)

print('Raster créé :', RASTER_PATH)
print(f'  Dimensions : {ROWS} × {COLS} pixels')
print(f'  NDVI min   : {ndvi.min():.3f} | max : {ndvi.max():.3f} | mean : {ndvi.mean():.3f}')

### 4.2 Exercice 4b — Lecture et inspection du raster

En suivant le CM §4.1, utilisez `rasterio.open()` pour inspecter le raster créé :

1. Affichez : CRS, résolution (`res`), dimensions (`shape`), nombre de bandes (`count`), nodata
2. Lisez la bande 1 avec `.read(1)` — notez que l'indice commence à **1**
3. Calculez et affichez : min, max, moyenne du NDVI
4. Expliquez pourquoi `.astype(float)` est essentiel avant le calcul du NDVI
   *(Indice : que se passe-t-il si on divise deux entiers uint16 en Python/NumPy ?)*

In [ ]:
# ── Correction 4b.1 : Métadonnées ──
with rasterio.open(RASTER_PATH) as src:
    print('CRS       :', src.crs)
    print('Résolution:', src.res, '(degrés/pixel)')
    print('Dimensions:', src.height, '×', src.width, 'pixels')
    print('Nb bandes :', src.count)
    print('NoData    :', src.nodata)
    print('Transform :', src.transform)
    transform_saved = src.transform

# ── Correction 4b.2 : Lecture bande 1 ──
with rasterio.open(RASTER_PATH) as src:
    # Important : .astype(float) avant tout calcul
    ndvi_lu = src.read(1).astype(float)

# ── Correction 4b.3 : Statistiques ──
# Masquer le nodata avant le calcul
ndvi_valid = ndvi_lu[ndvi_lu != NODATA]
print(f'NDVI min  : {ndvi_valid.min():.3f}')
print(f'NDVI max  : {ndvi_valid.max():.3f}')
print(f'NDVI mean : {ndvi_valid.mean():.3f}')

# Visualisation rapide
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(ndvi_lu, cmap='RdYlGn', vmin=0, vmax=0.7,
               extent=[BBOX[0], BBOX[2], BBOX[1], BBOX[3]])
plt.colorbar(im, ax=ax, label='NDVI', shrink=0.8)
ax.set_title('NDVI synthétique — Burkina Faso\n(gradient sahélien Nord-Sud)', fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig('M1_ndvi_raster.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Correction 4b.4 : Explication astype(float) ──
# Les images Sentinel-2 brutes sont encodées en uint16 (entiers 16 bits).
# Diviser deux uint16 en NumPy produit un entier (division entière) : 3 // 5 = 0.
# Résultat : NDVI = 0 ou 1 partout — faux et difficile à détecter !
# .astype(float) convertit en float64 → la division produit un flottant correct.
# 💡 Démonstration :
a = np.array([100], dtype=np.uint16)
b = np.array([300], dtype=np.uint16)
print('Division uint16 :', (b - a) / (b + a))     # → 0  (FAUX)
print('Division float  :', (b.astype(float) - a.astype(float)) / (b.astype(float) + a.astype(float)))  # → 0.5 (CORRECT)

### 4.3 Exercice 4c — NDVI moyen par région

Calculez le **NDVI moyen de chaque région** en utilisant `rasterstats.zonal_stats`.
Ajoutez la colonne `ndvi_mean` au GeoDataFrame `gdf`.

> 💡 `rasterstats` effectue automatiquement les statistiques zonales :
> pour chaque polygone de `gdf`, il agrège les pixels du raster qui tombent dedans.

In [ ]:
from rasterstats import zonal_stats

# ── Correction 4c ──
stats = zonal_stats(
    gdf,                  # vecteur (polygones)
    RASTER_PATH,          # raster source
    stats=['mean', 'min', 'max'],
    nodata=NODATA
)

gdf['ndvi_mean'] = [s['mean'] for s in stats]
gdf['ndvi_min']  = [s['min']  for s in stats]
gdf['ndvi_max']  = [s['max']  for s in stats]

print('=== NDVI moyen par région (croissant → Nord aride en premier) ===')
print(gdf[['region', 'ndvi_mean', 'ndvi_min', 'ndvi_max']]
        .sort_values('ndvi_mean')
        .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

# 💡 zonal_stats fait la reprojection automatiquement si vecteur et raster
# sont dans le même CRS. Ici tous deux en EPSG:4326 → OK.
# Pour un vrai raster Sentinel-2 (EPSG:32630), reprojeter gdf en 32630
# OU utiliser all_touched=True pour les petits polygones.

---
## Partie 5 — Cartographie Statique avec Contextily

> **Référence CM** : §6.1 Carte rapide avec `.plot()`, §6.2 Fond de carte Contextily

Le CM présente deux niveaux de visualisation statique :
1. `.plot()` direct sur un GeoDataFrame (rapide, pour le diagnostic)
2. `contextily` pour ajouter un fond de carte OpenStreetMap (livrable professionnel)

> ⚠️ **Rappel CM §6.2** : Contextily requiert **EPSG:3857** (Web Mercator).
> Il faut reprojeter le GeoDataFrame avant d'appeler `ctx.add_basemap()`.

### Exercice 5 — Deux cartes côte à côte

Créez une figure avec deux axes :
1. **Axe gauche** : Choroplèthe de la **population** avec `scheme='quantiles'`, k=4
   - Titre, légende, `axis('off')`, annotations des noms de régions
2. **Axe droit** : Fond de carte Contextily + points d'eau colorés par type
   - Reprojeter en EPSG:3857 avant `ctx.add_basemap()`
   - Forages en bleu, Puits en orange

In [ ]:
import contextily as ctx

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Correction 5.1 : Choroplèthe population ──
ax1 = axes[0]
gdf.plot(
    column='population',
    cmap='YlOrRd',
    scheme='quantiles',   # Classification en quantiles (comme dans le CM §6.1)
    k=4,
    legend=True,
    legend_kwds={'title': 'Population (hab.)', 'fmt': '{:,.0f}'},
    edgecolor='white',
    linewidth=0.8,
    ax=ax1
)
ax1.set_title('Population par Région\nBurkina Faso (INSD)', fontsize=12, fontweight='bold')
ax1.axis('off')

# Annotations noms de régions
for _, row in gdf.iterrows():
    c = row.geometry.centroid
    # Abréger pour lisibilité
    nom_court = row['region'].replace('Boucle du Mouhoun', 'B.Mouhoun')
    ax1.annotate(nom_court, xy=(c.x, c.y), ha='center', va='center',
                 fontsize=6.5, fontweight='bold', color='#2c3e50')

# ── Correction 5.2 : Contextily + points d'eau ──
ax2 = axes[1]

# Reprojection en EPSG:3857 (obligatoire pour Contextily)
gdf_web   = gdf.to_crs(epsg=3857)
puits_web = puits.to_crs(epsg=3857)

# Fond de régions semi-transparent
gdf_web.plot(ax=ax2, alpha=0.25, edgecolor='white', color='#3498db', linewidth=0.8)

# Points d'eau colorés par type
couleurs_type = {'Forage': '#2980b9', 'Puits': '#e67e22'}
for ptype, color in couleurs_type.items():
    subset = puits_web[puits_web['type'] == ptype]
    subset.plot(ax=ax2, color=color, markersize=30, label=ptype, zorder=5)

# Fond de carte OpenStreetMap
ctx.add_basemap(ax2, source=ctx.providers.OpenStreetMap.Mapnik, zoom=7)

ax2.set_title('Points d\'eau (forages & puits)\navec fond OpenStreetMap', fontsize=12, fontweight='bold')
ax2.legend(loc='lower left', fontsize=9)
ax2.axis('off')

plt.tight_layout()
plt.savefig('M1_cartes_statiques.png', dpi=150, bbox_inches='tight')
plt.show()
print('Carte sauvegardée : M1_cartes_statiques.png')

# 💡 Contextily télécharge les tuiles depuis internet au moment de l'exécution.
# Si la connexion est lente, utiliser source=ctx.providers.CartoDB.Positron
# (tuiles plus légères). Le paramètre zoom= contrôle la résolution :
# zoom=6 → BF entier en faible résolution, zoom=9 → détail local.

---
## Partie 6 — Carte Interactive avec Folium

> **Référence CM** : §6.3 Carte interactive avec Folium

Le CM présente Folium comme l'outil de livrable web : *idéal pour des interfaces
destinées à des non-techniciens ou des décideurs*.

### Exercice 6 — Tableau de bord interactif

Créez une carte Folium avec **3 couches** :

1. **Choroplèthe** de la population par région (`folium.Choropleth`)
2. **Tooltips** sur les régions affichant : Région, Chef-lieu, Population,
   Nb points d'eau, NDVI moyen
3. **Marqueurs** pour les points d'eau — icône différente selon le type :
   - Forage : `icon='tint'` (bleu)
   - Puits : `icon='circle'` (orange)

> ⚠️ **Rappel CM** : Folium travaille toujours en **EPSG:4326** (WGS84).
> Ne jamais lui passer un GeoDataFrame en EPSG:32630 ou 3857.

> 💡 **Bonne pratique popup** : construire le HTML par concaténation de strings —
> jamais avec des f-strings triple-guillemets imbriqués dans du code notebook.

In [ ]:
# ── Correction 6.1 : Initialisation ──
m = folium.Map(
    location=[12.37, -1.54],
    zoom_start=6,
    tiles='CartoDB positron'
)

# ── Correction 6.2 : Choroplèthe population ──
geojson_str = gdf.to_json()

folium.Choropleth(
    geo_data=geojson_str,
    name='Population',
    data=gdf,
    columns=['region', 'population'],
    key_on='feature.properties.region',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.8,
    legend_name='Population par région',
).add_to(m)

# ── Correction 6.3 : Tooltips ──
folium.GeoJson(
    geojson_str,
    name='Infos régions',
    style_function=lambda x: {'fillOpacity': 0, 'color': 'transparent'},
    tooltip=folium.GeoJsonTooltip(
        fields=['region', 'chef_lieu', 'population', 'nb_points_eau', 'ndvi_mean'],
        aliases=['Région', 'Chef-lieu', 'Population', 'Points d\'eau', 'NDVI moyen'],
        localize=True
    )
).add_to(m)

# ── Correction 6.4 : Marqueurs points d'eau ──
fg_puits = folium.FeatureGroup(name='Points d\'eau')

config_type = {
    'Forage': ('blue',   'tint'),
    'Puits':  ('orange', 'circle'),
}

for _, row in puits.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x
    ptype = row['type']
    color, icon = config_type.get(ptype, ('gray', 'info-sign'))

    # HTML popup sans f-string triple-guillemets
    popup_html = ('<div style="font-family:Arial;width:150px">'
                 + '<b>' + row['nom_point'] + '</b><br>'
                 + 'Type : ' + ptype
                 + '</div>')

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=160),
        icon=folium.Icon(color=color, icon=icon, prefix='glyphicon'),
        tooltip=row['nom_point'] + ' (' + ptype + ')'
    ).add_to(fg_puits)

fg_puits.add_to(m)

# ── Correction 6.5 : Finalisation ──
folium.LayerControl().add_to(m)
m.save('M1_carte_interactive.html')
print('✅ Carte interactive sauvegardée : M1_carte_interactive.html')
m

# 💡 key_on='feature.properties.region' doit correspondre EXACTEMENT
# au nom de la propriété dans le GeoJSON — une casse incorrecte
# provoque un Choropleth entièrement gris sans message d'erreur.

---
## ✅ Correction complète — Module 1

### Tableau de bord des bonnes pratiques

| Piège fréquent | Bonne pratique CM |
|----------------|-------------------|
| `.area` en EPSG:4326 → degrés² | Toujours `.to_crs(32630)` avant `.area` |
| CRS différents avant `sjoin` | `assert gdf.crs == puits.crs` |
| `src.read(0)` → IndexError Rasterio | Les bandes commencent à **1** |
| Division uint16 → NDVI = 0 | `.astype(float)` avant calcul |
| Contextily en EPSG:4326 → décalage | Toujours `.to_crs(3857)` avant |
| Choropleth Folium gris | `key_on` doit correspondre exactement |
| f-string triple-guillemets dans popup | Concaténation de strings |

### Note sur les données
Les géométries rectangulaires et le raster synthétique sont des approximations pédagogiques.
Les vrais shapefiles (Drive `1MHY8PprO8_RIyGIh_-1Hz5ybD7GJULFH`) s'intègrent
en remplaçant le bloc de chargement GeoJSON — tout le reste du TP fonctionne sans modification.

**Prochain module** : Feature Engineering Spatial — normalisation, IVSE, autocorrélation spatiale.